# General

In [ ]:
from datasets import load_dataset, Dataset
import sys, os
os.environ["ROOT_DIR"] = "PUPPET/experimental/"
for sm in ["lm-watermarking", "WaterBench"]:
    add_path = os.path.join(os.environ["ROOT_DIR"], "submodules", sm)
    sys.path.append(add_path)

In [ ]:
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{'punkt' if pkg=='punkt' else 'punkt_tab'}/english")
        print(f"Found {pkg} in nltk.")
    except LookupError:
        print(f"Not found {pkg} in nltk, so downloading...")
        nltk.download(pkg, quiet=True)

# ELI-5

## Preprocess

In [ ]:
# Based on WaterBench/process/download.py (https://github.com/THU-KEG/WaterBench/blob/8f3d779d66518a7b90ce1aad1fabaeb13cfca548/process/download.py#L122),
# they use eli-5 dataset from HC3 (https://huggingface.co/datasets/Hello-SimpleAI/HC3).
orig_ds = load_dataset("Hello-SimpleAI/HC3", "reddit_eli5", split="train")

In [ ]:
# Length filter: based on WaterBench/process/download.py (https://github.com/THU-KEG/WaterBench/blob/8f3d779d66518a7b90ce1aad1fabaeb13cfca548/process/download.py#L127-L131)
from nltk.tokenize import word_tokenize
def _add_average_token_length_of_human_answers(sample: dict) -> dict:
    sample["output_length"] = sum([ len(word_tokenize(x)) for x in sample["human_answers"] ]) / len(sample["human_answers"])
    return sample
orig_ds_with_len_col = orig_ds.map(_add_average_token_length_of_human_answers)
not2long_orig_ds = orig_ds_with_len_col.filter(lambda x: x["output_length"] <= 300)
print(f"Removed {len(orig_ds_with_len_col) - len(not2long_orig_ds):,} samples, and remaining {len(not2long_orig_ds):,} [{len(not2long_orig_ds)/len(orig_ds_with_len_col)*100:.2f}%] samples.")

In [ ]:
# Duplicate filter: remove samples used in WaterBench as a test set.
dedup_orig_ds = not2long_orig_ds.filter(lambda x: x["question"] not in wb_ds["input"])
print(f"Removed {len(not2long_orig_ds) - len(dedup_orig_ds):,} samples, and remaining {len(dedup_orig_ds):,} [{len(dedup_orig_ds)/len(not2long_orig_ds)*100:.2f}%] samples.")

In [ ]:
# Sort by the length following WaterBench/process/download.py (https://github.com/THU-KEG/WaterBench/blob/8f3d779d66518a7b90ce1aad1fabaeb13cfca548/process/download.py#L133).
sorted_orig_ds = dedup_orig_ds.sort("output_length", reverse=True)
print(f"First: {sorted_orig_ds['output_length'][0]:.0f}, Last: {sorted_orig_ds['output_length'][-1]:.0f}")

In [ ]:
print(f"Preprocessed dataset: {len(sorted_orig_ds):,} samples.")

In [ ]:
# Match columns to those of WaterBench/data/WaterBench/2-1_longform_qa.jsonl.
def _add_required_columns(sample: dict) -> dict:
    sample["input_length"] = len(word_tokenize(sample["question"]))
    sample["length"] = sample["input_length"] + sample["output_length"]
    sample["all_classes"] = "null"
    sample["language"] = "en"
    sample["dataset"] = "longform_qa"
    sample["context"] = ""
    return sample
preprocessed_ds = sorted_orig_ds.map(_add_required_columns)
preprocessed_ds = preprocessed_ds.rename_columns({"id": "_id", "question": "input", "human_answers": "outputs"})
preprocessed_ds = preprocessed_ds.remove_columns(["chatgpt_answers"])
preprocessed_ds[0]

In [ ]:
# Save the preprocessed dataset.
preprocessed_ds.save_to_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/eli5_preprocessed_202510281307"))

In [ ]:
from datasets import load_from_disk

preprocessed_dataset = load_from_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/eli5_preprocessed_202510281307"))

In [ ]:
import random

indices = list(range(len(preprocessed_dataset)))

random.seed(42)
random.shuffle(indices)

print(indices)

In [ ]:
num_samples = 5000

random_sampled_dataset = preprocessed_dataset.select(indices[:num_samples])
random_sampled_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], f"dpo_data/data/eli5_preprocessed_202510281307_random_{num_samples}"))

## Split

In [ ]:
gen_path = "PUPPET/experimental/vllm_gen/data/for_dpo_data/different_sizes/generation_vllm_longform_qa_qwen3_14b_random_5000_2026_0321_065144.jsonl"
gen_ds = load_dataset("json", data_files=gen_path, split="train")

evl_path = "PUPPET/experimental/evaluation/results/for_dpo_data/different_sizes/evaluation_vllm_longform_qa_qwen3_14b_5000_2026_0321_211544.jsonl"
evl_ds = load_dataset("json", data_files=evl_path, split="train")

In [ ]:
import yaml
with open("PUPPET/experimental/dpo_data/configs/eli5_detect_rouge_standarzied_5000.yaml", "r") as f:
    dataset_config = yaml.safe_load(f)

In [ ]:
from nltk.tokenize import word_tokenize

def make_pairs(generation_dataset: Dataset, evaluation_dataset: Dataset, config: dict) -> Dataset:
    def __preprocess_scores(evaluation_sample: Dataset) -> Dataset:
        def __standardize(x):
            m = sum(x)/len(x)
            s = (sum((v-m)**2 for v in x)/len(x))**0.5
            return [(v-m)/(s or 1) for v in x]
        procs = {
            "standardize": __standardize,
        }
        for score_name in config["preprocess"].keys():
            evaluation_sample[score_name] = procs[config["preprocess"][score_name]](evaluation_sample[score_name])
        return evaluation_sample
    
    def __calc_weighted_scores(evaluation_sample: Dataset) -> list[float]:
        weighted_scores = []
        for evaluation_idx in range(len(evaluation_sample[config["target_fields"]["evaluation"]])):
            weighted_scores.append(
                sum(
                    weight*evaluation_sample[score_name][evaluation_idx]
                    for score_name, weight in config["score_weights"].items()
                )
            )
        return weighted_scores

    def __make_pair(weighted_scores: list[float]) -> dict[str, int]:
        if config["pair_strategy"] == "best_and_worst":
            best_idx = weighted_scores.index(max(weighted_scores))
            worst_idx = weighted_scores.index(min(weighted_scores))
            return {"chosen": best_idx, "rejected": worst_idx}
        else:
            raise ValueError(f"Invalid pair strategy: {config['pair_strategy']}")

    def __add_pair_indexes(evaluation_sample: Dataset) -> dict:
        evaluation_sample = __preprocess_scores(evaluation_sample)
        evaluation_sample["weighted_scores"] = __calc_weighted_scores(evaluation_sample)
        evaluation_sample["pair_indexes"] = __make_pair(evaluation_sample["weighted_scores"])
        return evaluation_sample
    
    def __add_pair_columns(generation_sample: Dataset, sample_idx: int, weighted_scores_list: list[float], pair_indexes_list: list[dict[str, int]]) -> dict:
        chosen_idx, rejected_idx = pair_indexes_list[sample_idx]["chosen"], pair_indexes_list[sample_idx]["rejected"]

        generation_sample["chosen"] = generation_sample[config["target_fields"]["generation"]][chosen_idx]
        generation_sample["rejected"] = generation_sample[config["target_fields"]["generation"]][rejected_idx]
       
        generation_sample["chosen_length"] = len(word_tokenize(generation_sample["chosen"]))
        generation_sample["rejected_length"] = len(word_tokenize(generation_sample["rejected"]))

        generation_sample["chosen_score"] = weighted_scores_list[sample_idx][chosen_idx]
        generation_sample["rejected_score"] = weighted_scores_list[sample_idx][rejected_idx]

        generation_sample["pair_indexes"] = pair_indexes_list[sample_idx]
        return generation_sample

    processed_evaluation_dataset = evaluation_dataset.map(__add_pair_indexes)
    processed_generation_dataset = generation_dataset.map(__add_pair_columns, with_indices=True, fn_kwargs={"pair_indexes_list": processed_evaluation_dataset["pair_indexes"], "weighted_scores_list": processed_evaluation_dataset["weighted_scores"]})

    return processed_generation_dataset, processed_evaluation_dataset

In [ ]:
proc_gen_ds, proc_evl_ds = make_pairs(gen_ds, evl_ds, dataset_config)

In [ ]:
from nltk.tokenize import word_tokenize

def convert_to_dpo_format(sample: dict) -> dict:
    def __convert_to_conversation_format(role: str, text: str) -> dict[str, str]:
        return [{"role": role, "content": text}]
    sample["prompt"] = __convert_to_conversation_format("user", sample["filled_prompt"])
    sample["chosen"] = __convert_to_conversation_format("assistant", sample["chosen"])
    sample["rejected"] = __convert_to_conversation_format("assistant", sample["rejected"])
    return sample

proc_gen_ds = proc_gen_ds.map(convert_to_dpo_format)
proc_gen_ds[0]

In [ ]:
import json
dataset_config_dump = json.dumps(dataset_config)
generation_config = gen_ds["config"][0]
evaluation_config = evl_ds["config"][0]

In [ ]:
def set_configs(sample: dict, dataset_config: str, generation_config: str, evaluation_config: str) -> dict:
    sample["dataset_config"] = dataset_config
    sample["generation_config"] = generation_config
    sample["evaluation_config"] = evaluation_config
    return sample

proc_gen_ds = proc_gen_ds.map(set_configs, fn_kwargs={"dataset_config": dataset_config_dump, "generation_config": generation_config, "evaluation_config": evaluation_config})
proc_gen_ds[0]

In [ ]:
dpo_ds = proc_gen_ds.remove_columns(list(set(proc_gen_ds.column_names) - set(dataset_config["use_columns"])))

In [ ]:
train_size, test_size = dataset_config["data_balance"]["for_training"], dataset_config["data_balance"]["for_evaluation"]
assert train_size + test_size == 1

if test_size > 0:
    split_dpo_ds = dpo_ds.train_test_split(train_size=train_size, seed=dataset_config["data_balance"]["seed"])
    print(f"train_size: {len(split_dpo_ds['train']):,}, test_size: {len(split_dpo_ds['test']):,}")
else:
    split_dpo_ds = dpo_ds

In [ ]:
from datetime import datetime

model_name = "qwen3"
dataset_name = f"{dataset_config['dataset_name']}_{model_name}_{datetime.now().strftime('%Y%m%d%H%M')}"

split_dpo_ds.save_to_disk(os.path.join(
    os.environ["ROOT_DIR"], 
    "dpo_data/data/",
    dataset_name
    ))

## Random Sample

In [ ]:
from datasets import load_from_disk

preprocessed_dataset_train = load_from_disk("PUPPET/experimental/dpo_data/data/eli5_preprocessed_202510281307")

In [ ]:
import random

indices = list(range(len(preprocessed_dataset_train)))

random.seed(42)
random.shuffle(indices)

print(indices)

In [ ]:
num_samples = 5000

random_sampled_dataset = preprocessed_dataset_train.select(indices[:num_samples])
original_dataset_name = "eli5_detect_rouge_standarzied_mage_202602171321"
dataset_name = f"{original_dataset_name}_random_{num_samples}"
random_sampled_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], f"dpo_data/data/{dataset_name}"))

# MultiNews

## Preprocess

In [ ]:
# Based on WaterBench/process/download.py (https://github.com/THU-KEG/WaterBench/blob/8f3d779d66518a7b90ce1aad1fabaeb13cfca548/process/download.py#L78-L89),

from huggingface_hub import hf_hub_download

train_src_path = hf_hub_download(
    repo_id="multi_news",
    filename="data/train.src.cleaned",
    repo_type="dataset",
)

train_tgt_path = hf_hub_download(
    repo_id="multi_news",
    filename="data/train.tgt",
    repo_type="dataset",
)

with open(train_src_path, encoding="utf-8") as src_f, open(train_tgt_path, encoding="utf-8") as tgt_f:
    raw_data = [
        {
            "concatenated_document": src_line,
            "summary": tgt_line
        }
        for src_line, tgt_line in zip(src_f, tgt_f)
    ]

In [ ]:
SPLIT_SYMBOL = "|||||"
def split_document(concatenated_document: str) -> list[str]:
    split_docs = concatenated_document.split(SPLIT_SYMBOL)
    return list(map(str.strip, split_docs))

In [ ]:
FORMAT_TEMPLATE = "Passage {doc_index}:\n{doc}"
def format_document(concatenated_document: str) -> str:
    split_docs = split_document(concatenated_document)
    formatted_docs = [FORMAT_TEMPLATE.format(doc_index=i+1, doc=doc) for i, doc in enumerate(split_docs)]
    return "\n".join(formatted_docs).strip()

In [ ]:
SUMMARY_PREFIX = "– "
def remove_prefix(summary: str) -> str:
    return summary.strip().lstrip(SUMMARY_PREFIX)

In [ ]:
USING_COLUMNS = ['context', 'answers']
def drop_useless_columns(sample: dict) -> dict:
    useless_columns = set(sample.keys()) - set(USING_COLUMNS)
    for col in useless_columns:
        sample.pop(col)
    return sample

In [ ]:
def convert_to_context(sample: dict) -> dict:
    processed_sample = sample.copy()
    processed_sample['context'] = format_document(processed_sample['concatenated_document'])
    processed_sample['answers'] = [remove_prefix(processed_sample['summary'])]  # 'answers' should have been 'outputs'
    processed_sample = drop_useless_columns(processed_sample)
    return processed_sample

In [ ]:
processed_data = list(map(convert_to_context, raw_data))

In [ ]:
from datasets import Dataset

processed_dataset = Dataset.from_list(processed_data)

In [ ]:
# Save the preprocessed dataset.
processed_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/multinews_preprocessed_202512031434"))

## Random Sample

In [ ]:
from datasets import load_from_disk

preprocessed_dataset = load_from_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/multinews_preprocessed_202512031434"))

In [ ]:
import random

indices = list(range(len(preprocessed_dataset)))

random.seed(42)
random.shuffle(indices)

print(indices)

In [ ]:
num_samples = 5000

random_sampled_dataset = preprocessed_dataset.select(indices[:num_samples])
random_sampled_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], f"dpo_data/data/multinews_preprocessed_202512031434_random_{num_samples}"))

## Spilit

In [ ]:
gen_path = "PUPPET/experimental/vllm_gen/data/for_dpo_data/different_sizes/generation_vllm_multi_news_qwen3_14b_random_5000_2026_0321_065833.jsonl"
gen_ds = load_dataset('json', data_files=gen_path, split="train")

evl_path = "PUPPET/experimental/evaluation/results/for_dpo_data/different_sizes/evaluation_vllm_multi_news_qwen3_14b_5000_2026_0321_211444.jsonl"
evl_ds = load_dataset('json', data_files=evl_path, split="train")

In [ ]:
import yaml
with open("PUPPET/experimental/dpo_data/configs/multinews_detect_rouge_standarzied_5000.yaml", "r") as f:
    dataset_config = yaml.safe_load(f)

In [ ]:
from nltk.tokenize import word_tokenize

def make_pairs(generation_dataset: Dataset, evaluation_dataset: Dataset, config: dict) -> Dataset:
    def __preprocess_scores(evaluation_sample: Dataset) -> Dataset:
        def __standardize(x):
            m = sum(x)/len(x)
            s = (sum((v-m)**2 for v in x)/len(x))**0.5
            return [(v-m)/(s or 1) for v in x]
        procs = {
            "standardize": __standardize,
        }
        for score_name in config["preprocess"].keys():
            evaluation_sample[score_name] = procs[config["preprocess"][score_name]](evaluation_sample[score_name])
        return evaluation_sample
    
    def __calc_weighted_scores(evaluation_sample: Dataset) -> list[float]:
        weighted_scores = []
        for evaluation_idx in range(len(evaluation_sample[config["target_fields"]["evaluation"]])):
            weighted_scores.append(
                sum(
                    weight*evaluation_sample[score_name][evaluation_idx]
                    for score_name, weight in config["score_weights"].items()
                )
            )
        return weighted_scores

    def __make_pair(weighted_scores: list[float]) -> dict[str, int]:
        if config["pair_strategy"] == "best_and_worst":
            best_idx = weighted_scores.index(max(weighted_scores))
            worst_idx = weighted_scores.index(min(weighted_scores))
            return {"chosen": best_idx, "rejected": worst_idx}
        else:
            raise ValueError(f"Invalid pair strategy: {config['pair_strategy']}")

    def __add_pair_indexes(evaluation_sample: Dataset) -> dict:
        evaluation_sample = __preprocess_scores(evaluation_sample)
        evaluation_sample["weighted_scores"] = __calc_weighted_scores(evaluation_sample)
        evaluation_sample["pair_indexes"] = __make_pair(evaluation_sample["weighted_scores"])
        return evaluation_sample
    
    def __add_pair_columns(generation_sample: Dataset, sample_idx: int, weighted_scores_list: list[float], pair_indexes_list: list[dict[str, int]]) -> dict:
        chosen_idx, rejected_idx = pair_indexes_list[sample_idx]["chosen"], pair_indexes_list[sample_idx]["rejected"]

        generation_sample["chosen"] = generation_sample[config["target_fields"]["generation"]][chosen_idx]
        generation_sample["rejected"] = generation_sample[config["target_fields"]["generation"]][rejected_idx]
       
        generation_sample["chosen_length"] = len(word_tokenize(generation_sample["chosen"]))
        generation_sample["rejected_length"] = len(word_tokenize(generation_sample["rejected"]))

        generation_sample["chosen_score"] = weighted_scores_list[sample_idx][chosen_idx]
        generation_sample["rejected_score"] = weighted_scores_list[sample_idx][rejected_idx]

        generation_sample["pair_indexes"] = pair_indexes_list[sample_idx]
        return generation_sample

    processed_evaluation_dataset = evaluation_dataset.map(__add_pair_indexes)
    processed_generation_dataset = generation_dataset.map(__add_pair_columns, with_indices=True, fn_kwargs={"pair_indexes_list": processed_evaluation_dataset["pair_indexes"], "weighted_scores_list": processed_evaluation_dataset["weighted_scores"]})

    return processed_generation_dataset, processed_evaluation_dataset

In [ ]:
proc_gen_ds, proc_evl_ds = make_pairs(gen_ds, evl_ds, dataset_config)

In [ ]:
from nltk.tokenize import word_tokenize

def convert_to_dpo_format(sample: dict) -> dict:
    def __convert_to_conversation_format(role: str, text: str) -> dict[str, str]:
        return [{"role": role, "content": text}]
    sample["prompt"] = __convert_to_conversation_format("user", sample["filled_prompt"])
    sample["chosen"] = __convert_to_conversation_format("assistant", sample["chosen"])
    sample["rejected"] = __convert_to_conversation_format("assistant", sample["rejected"])
    return sample

proc_gen_ds = proc_gen_ds.map(convert_to_dpo_format)
proc_gen_ds[0]

In [ ]:
import json
dataset_config_dump = json.dumps(dataset_config)
generation_config = gen_ds["config"][0]
evaluation_config = evl_ds["config"][0]

In [ ]:
def set_configs(sample: dict, dataset_config: str, generation_config: str, evaluation_config: str) -> dict:
    sample["dataset_config"] = dataset_config
    sample["generation_config"] = generation_config
    sample["evaluation_config"] = evaluation_config
    return sample

proc_gen_ds = proc_gen_ds.map(set_configs, fn_kwargs={"dataset_config": dataset_config_dump, "generation_config": generation_config, "evaluation_config": evaluation_config})
proc_gen_ds[0]


In [ ]:
dpo_ds = proc_gen_ds.remove_columns(list(set(proc_gen_ds.column_names) - set(dataset_config["use_columns"])))

In [ ]:
train_size, test_size = dataset_config["data_balance"]["for_training"], dataset_config["data_balance"]["for_evaluation"]
assert train_size + test_size == 1

if test_size > 0:
    split_dpo_ds = dpo_ds.train_test_split(train_size=train_size, seed=dataset_config["data_balance"]["seed"])
    print(f"train_size: {len(split_dpo_ds['train']):,}, test_size: {len(split_dpo_ds['test']):,}")
else:
    split_dpo_ds = dpo_ds

In [ ]:
from datetime import datetime

model_name = "qwen3"
dataset_name = f"{dataset_config['dataset_name']}_{model_name}_{datetime.now().strftime('%Y%m%d%H%M')}"

split_dpo_ds.save_to_disk(os.path.join(
    os.environ["ROOT_DIR"], 
    "dpo_data/data/",
    dataset_name
    ))

## Random Sample

In [ ]:
from datasets import load_from_disk

preprocessed_dataset_train = load_from_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/multinews_detect_rouge_standarzied_5000_202512041009_llama3"))

In [ ]:
import random

indices = list(range(len(preprocessed_dataset_train)))

random.seed(42)
random.shuffle(indices)

print(indices)

num_samples = 625

In [ ]:
random_sampled_dataset = preprocessed_dataset_train.select(indices[:num_samples])

In [ ]:
random_sampled_dataset = preprocessed_dataset_train.select(indices[:num_samples])
dataset_name = f"multinews_detect_rouge_standarzied_5000_202512041009_llama3_pick_{num_samples}"
random_sampled_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], f"dpo_data/data/{dataset_name}"))

## Preprocess

In [ ]:
import pickle

split = "test"
file_dir = f"PUPPET/experimental/submodules/OUTFOX/data/common/{split}"
file_paths = {
    "problem_statements": "{split}_problem_statements.pkl",
    "human_answers": "{split}_humans.pkl",
    "contexts": "{split}_contexts.pkl",
}
data = []

tmp_dct = {}
for file_name, file_path in file_paths.items():
    with open(os.path.join(file_dir, file_path.format(split=split)), "rb") as f:
        tmp_dct[file_name] = pickle.load(f)

data = [
    {
        "problem_statements": tmp_dct["problem_statements"][idx],
        "human_answers": tmp_dct["human_answers"][idx],
        "contexts": tmp_dct["contexts"][idx],
    }
    for idx in range(len(tmp_dct["problem_statements"]))
]

In [ ]:
import random

random.seed(42)

indices = list(range(len(data)))
random.shuffle(indices)
print(indices)

data_size = {
    "train": 5000,
    "test": 200,
}[split]

print(data_size)
data_random_sampled = [data[idx] for idx in indices[:data_size]]

In [ ]:
human_answers = [{'outputs': [sample['human_answers']]} for sample in data_random_sampled]
human_answers[0]

In [ ]:
import json
with open(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/outfox_preprocessed_random_200_test_202512161348_human_answers.jsonl"), "w") as f:
    for sample in human_answers:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

In [ ]:
USE_COLUMNS = ["context", "outputs"]
def remove_useless_columns(sample: dict) -> dict:
    useless_columns = set(sample.keys()) - set(USE_COLUMNS)
    for col in useless_columns:
        sample.pop(col)
    return sample

def convert_to_context(sample: dict) -> dict:
    processed_sample = sample.copy()
    processed_sample['context'] = sample["contexts"]
    processed_sample['outputs'] = sample['human_answers']
    processed_sample = remove_useless_columns(processed_sample)
    return processed_sample

In [ ]:
processed_data = list(map(convert_to_context, data_random_sampled))

In [ ]:
from datasets import Dataset

processed_dataset = Dataset.from_list(processed_data)

In [ ]:
# Save the preprocessed dataset.
processed_dataset.save_to_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/outfox_preprocessed_random_5000_202512161348"))

## Split

In [ ]:
gen_path = "PUPPET/experimental/vllm_gen/data/for_dpo_data/generation_vllm_outfox_llama3_random_5000_2025_1217_163808.jsonl"
gen_ds = load_dataset('json', data_files=gen_path, split="train")

evl_path = "PUPPET/experimental/evaluation/results/for_dpo_data/evaluation_vllm_outfox_llama3_5000_2026_0116_181358_logprobs_ielts.jsonl"
evl_ds = load_dataset('json', data_files=evl_path, split="train")

In [ ]:
import yaml
with open(os.path.join(os.environ["ROOT_DIR"], "dpo_data/configs/outfox_detect_laaj_standarzied_5000.yaml"), "r") as f:
    dataset_config = yaml.safe_load(f)

In [ ]:
from __future__ import annotations

from nltk.tokenize import word_tokenize
from datasets import Dataset


def make_pairs(generation_dataset: Dataset, evaluation_dataset: Dataset, config: dict) -> tuple[Dataset, Dataset]:
    """
    Each sample in evaluation_dataset has k candidates.
    For each sample:
      - Drop candidates whose any score == ignore_score
      - If remaining candidates < 2: raise ValueError
      - Select chosen/rejected from remaining candidates based on pair_strategy
      - Return pair indices in the ORIGINAL k-index space
    """

    # -----------------------------
    # Config helpers
    # -----------------------------
    def _score_names() -> list[str]:
        scores_cfg = config.get("scores", None)
        if not isinstance(scores_cfg, dict) or not scores_cfg:
            raise ValueError("config['scores'] must be a non-empty dict.")
        return list(scores_cfg.keys())

    def _score_weight(score_name: str) -> float:
        w = config["scores"][score_name].get("weight", None)
        if w is None:
            raise ValueError(f"config['scores']['{score_name}']['weight'] is required.")
        return float(w)

    def _score_preproc(score_name: str) -> str | None:
        # preprocessing may be omitted / None
        return config["scores"][score_name].get("preprocessing", None)

    # -----------------------------
    # Candidate filtering
    # -----------------------------
    def __get_valid_candidate_indices(evaluation_sample: dict) -> list[int]:
        ignore = config.get("ignore_score", None)
        if ignore is None:
            raise ValueError("config['ignore_score'] is required.")

        eval_field = config["target_fields"]["evaluation"]
        if eval_field not in evaluation_sample:
            raise ValueError(f"evaluation sample does not have field '{eval_field}'.")

        k = len(evaluation_sample[eval_field])
        if k <= 0:
            raise ValueError("Each sample must have at least 1 candidate.")

        score_fields = _score_names()

        valid: list[int] = []
        for i in range(k):
            bad = False
            for score_name in score_fields:
                if score_name not in evaluation_sample:
                    raise ValueError(f"evaluation sample missing score field '{score_name}'.")
                v = evaluation_sample[score_name][i]
                if v == ignore:
                    bad = True
                    break
            if not bad:
                valid.append(i)

        if len(valid) < 2:
            raise ValueError(
                f"Not enough valid candidates after ignoring ignore_score. "
                f"valid={len(valid)} (<2)."
            )
        return valid

    # -----------------------------
    # Preprocessing (standardize)
    # -----------------------------
    def __standardize(vals: list[float]) -> list[float]:
        m = sum(vals) / len(vals)
        s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
        denom = (s or 1.0)
        return [(v - m) / denom for v in vals]

    _PREPROCS = {
        "standardize": __standardize,
    }

    def __preprocess_scores_on_valid(evaluation_sample: dict, valid: list[int]) -> dict:
        """
        Apply preprocessing for each score_name to values ONLY on valid candidates.
        Store the processed score array back in original length k,
        filling invalid positions with None.
        """
        k = len(evaluation_sample[config["target_fields"]["evaluation"]])

        for score_name in _score_names():
            preproc = _score_preproc(score_name)
            if not preproc:
                # no preprocessing requested: still keep original values but None-out invalid to avoid accidental use
                full = [None] * k
                for i in valid:
                    full[i] = evaluation_sample[score_name][i]
                evaluation_sample[score_name] = full
                continue

            if preproc not in _PREPROCS:
                raise ValueError(f"Unknown preprocessing '{preproc}' for score '{score_name}'.")

            proc_fn = _PREPROCS[preproc]
            valid_vals = [float(evaluation_sample[score_name][i]) for i in valid]
            valid_vals_proc = proc_fn(valid_vals)

            full = [None] * k
            for i, vproc in zip(valid, valid_vals_proc):
                full[i] = vproc
            evaluation_sample[score_name] = full

        return evaluation_sample

    # -----------------------------
    # Weighted scores + pairing
    # -----------------------------
    def __calc_weighted_scores_full(evaluation_sample: dict, valid: list[int]) -> list[float | None]:
        """
        Return weighted_scores in original length k.
        invalid positions are None.
        """
        k = len(evaluation_sample[config["target_fields"]["evaluation"]])
        weighted: list[float | None] = [None] * k

        for i in valid:
            s = 0.0
            for score_name in _score_names():
                w = _score_weight(score_name)
                v = evaluation_sample[score_name][i]
                if v is None:
                    # should never happen for valid i
                    raise RuntimeError(f"Score '{score_name}' at valid index {i} is None.")
                s += w * float(v)
            weighted[i] = s

        return weighted

    def __make_pair_from_full(weighted_scores_full: list[float | None], valid: list[int]) -> dict[str, int]:
        if config.get("pair_strategy") == "best_and_worst":
            chosen_idx = max(valid, key=lambda i: weighted_scores_full[i])   # type: ignore[arg-type]
            rejected_idx = min(valid, key=lambda i: weighted_scores_full[i]) # type: ignore[arg-type]
            return {"chosen": chosen_idx, "rejected": rejected_idx}
        else:
            raise ValueError(f"Invalid pair strategy: {config.get('pair_strategy')}")

    def __add_pair_indexes(evaluation_sample: dict) -> dict:
        valid = __get_valid_candidate_indices(evaluation_sample)
        evaluation_sample = __preprocess_scores_on_valid(evaluation_sample, valid)

        evaluation_sample["weighted_scores"] = __calc_weighted_scores_full(evaluation_sample, valid)
        evaluation_sample["pair_indexes"] = __make_pair_from_full(evaluation_sample["weighted_scores"], valid)

        # optional for debugging / analysis
        evaluation_sample["valid_candidate_indices"] = valid
        return evaluation_sample

    # -----------------------------
    # Generation side: add pair columns
    # -----------------------------
    def __add_pair_columns(
        generation_sample: dict,
        sample_idx: int,
        weighted_scores_list: list[list[float | None]],
        pair_indexes_list: list[dict[str, int]],
    ) -> dict:
        chosen_idx = pair_indexes_list[sample_idx]["chosen"]
        rejected_idx = pair_indexes_list[sample_idx]["rejected"]

        gen_field = config["target_fields"]["generation"]
        gen_candidates = generation_sample[gen_field]

        generation_sample["chosen"] = gen_candidates[chosen_idx]
        generation_sample["rejected"] = gen_candidates[rejected_idx]

        # lengths
        if "prompt" in generation_sample:
            generation_sample["prompt_length"] = len(word_tokenize(generation_sample["prompt"]))
        generation_sample["chosen_length"] = len(word_tokenize(generation_sample["chosen"]))
        generation_sample["rejected_length"] = len(word_tokenize(generation_sample["rejected"]))

        # scores
        ws = weighted_scores_list[sample_idx]
        generation_sample["chosen_score"] = ws[chosen_idx]
        generation_sample["rejected_score"] = ws[rejected_idx]

        # indexes
        generation_sample["pair_indexes"] = pair_indexes_list[sample_idx]

        return generation_sample

    # -----------------------------
    # Run
    # -----------------------------
    processed_evaluation_dataset = evaluation_dataset.map(__add_pair_indexes)

    processed_generation_dataset = generation_dataset.map(
        __add_pair_columns,
        with_indices=True,
        fn_kwargs={
            "pair_indexes_list": processed_evaluation_dataset["pair_indexes"],
            "weighted_scores_list": processed_evaluation_dataset["weighted_scores"],
        },
    )

    return processed_generation_dataset, processed_evaluation_dataset

In [ ]:
proc_gen_ds, proc_evl_ds = make_pairs(gen_ds, evl_ds, dataset_config)

In [ ]:
from nltk.tokenize import word_tokenize

def convert_to_dpo_format(sample: dict) -> dict:
    def __convert_to_conversation_format(role: str, text: str) -> dict[str, str]:
        return [{"role": role, "content": text}]
    sample["prompt"] = __convert_to_conversation_format("user", sample["filled_prompt"])
    sample["chosen"] = __convert_to_conversation_format("assistant", sample["chosen"])
    sample["rejected"] = __convert_to_conversation_format("assistant", sample["rejected"])
    return sample

proc_gen_ds = proc_gen_ds.map(convert_to_dpo_format)
proc_gen_ds[0]

In [ ]:
import json
dataset_config_dump = json.dumps(dataset_config)
generation_config = gen_ds["config"][0]
evaluation_config = evl_ds["config"][0]

In [ ]:
def set_configs(sample: dict, dataset_config: str, generation_config: str, evaluation_config: str) -> dict:
    sample["dataset_config"] = dataset_config
    sample["generation_config"] = generation_config
    sample["evaluation_config"] = evaluation_config
    return sample

proc_gen_ds = proc_gen_ds.map(set_configs, fn_kwargs={"dataset_config": dataset_config_dump, "generation_config": generation_config, "evaluation_config": evaluation_config})
proc_gen_ds[0]


In [ ]:
dpo_ds = proc_gen_ds.remove_columns(list(set(proc_gen_ds.column_names) - set(dataset_config["use_columns"])))

In [ ]:
train_size, test_size = dataset_config["data_balance"]["for_training"], dataset_config["data_balance"]["for_evaluation"]
assert train_size + test_size == 1

if test_size > 0:
    split_dpo_ds = dpo_ds.train_test_split(train_size=train_size, seed=dataset_config["data_balance"]["seed"])
    print(f"train_size: {len(split_dpo_ds['train']):,}, test_size: {len(split_dpo_ds['test']):,}")
else:
    split_dpo_ds = dpo_ds

In [ ]:
from datetime import datetime

model_name = "llama3"
dataset_name = f"{dataset_config['dataset_name']}_{model_name}_{datetime.now().strftime('%Y%m%d%H%M')}"

split_dpo_ds.save_to_disk(os.path.join(
    os.environ["ROOT_DIR"], 
    "dpo_data/data/",
    dataset_name
    ))

# IELTS

## Preprocess

In [ ]:
from datasets import load_dataset

ds = load_dataset("chillies/IELTS-writing-task-2-evaluation")
ds_train, ds_test = ds["train"], ds["test"]

In [ ]:
import random

train_indices = list(range(len(ds_train)))
random.seed(42)
random.shuffle(train_indices)
ds_train_random = [ds_train[idx] for idx in train_indices[:5000]]

test_indices = list(range(len(ds_train)))
random.seed(42)
random.shuffle(test_indices)
ds_test_random = [ds_train[idx] for idx in test_indices[:200]]

In [ ]:
def format_sample(sample: dict) -> dict:
    sample['context'] = sample['prompt']
    sample['outputs'] = sample['essay']
    for useless_key in ['prompt', 'essay', 'evaluation', 'band']:
        sample.pop(useless_key)
    return sample

ds_train_formatted = []
for sample in ds_train_random:
    ds_train_formatted.append(format_sample(sample))

ds_test_formatted = []
for sample in ds_test_random:
    ds_test_formatted.append(format_sample(sample))

In [ ]:
from datasets import Dataset

ds_train_preprocessed = Dataset.from_list(ds_train_formatted)
ds_test_preprocessed = Dataset.from_list(ds_test_formatted)

In [ ]:
ds_train_preprocessed.save_to_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/ielts_preprocessed_random_5000_202601161730"))
ds_test_preprocessed.save_to_disk(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/ielts_preprocessed_random_200_test_202601161730"))

In [ ]:
import json
with open(os.path.join(os.environ["ROOT_DIR"], "dpo_data/data/ielts_preprocessed_random_200_test_202601161730_human_answers.jsonl"), "w") as f:
    for sample in human_answers:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

## Split

In [ ]:
gen_path = "PUPPET/experimental/vllm_gen/data/for_dpo_data/different_sizes/generation_vllm_ielts_qwen3_4b_random_5000_2026_0321_065634.jsonl"
gen_ds = load_dataset('json', data_files=gen_path, split="train")

evl_path = "PUPPET/experimental/evaluation/results/for_dpo_data/different_sizes/evaluation_vllm_ielts_qwen3_4b_5000_2026_0321_211257.jsonl"
evl_ds = load_dataset('json', data_files=evl_path, split="train")

In [ ]:
import yaml
with open("PUPPET/experimental/dpo_data/configs/ielts_detect_laaj_standarzied_5000.yaml", "r") as f:
    dataset_config = yaml.safe_load(f)

In [ ]:
from __future__ import annotations

from nltk.tokenize import word_tokenize
from datasets import Dataset


def make_pairs(generation_dataset: Dataset, evaluation_dataset: Dataset, config: dict) -> tuple[Dataset, Dataset]:
    """
    Each sample in evaluation_dataset has k candidates.
    For each sample:
      - Drop candidates whose any score == ignore_score
      - If remaining candidates < 2: raise ValueError
      - Select chosen/rejected from remaining candidates based on pair_strategy
      - Return pair indices in the ORIGINAL k-index space
    """

    # -----------------------------
    # Config helpers
    # -----------------------------
    def _score_names() -> list[str]:
        scores_cfg = config.get("scores", None)
        if not isinstance(scores_cfg, dict) or not scores_cfg:
            raise ValueError("config['scores'] must be a non-empty dict.")
        return list(scores_cfg.keys())

    def _score_weight(score_name: str) -> float:
        w = config["scores"][score_name].get("weight", None)
        if w is None:
            raise ValueError(f"config['scores']['{score_name}']['weight'] is required.")
        return float(w)

    def _score_preproc(score_name: str) -> str | None:
        # preprocessing may be omitted / None
        return config["scores"][score_name].get("preprocessing", None)

    # -----------------------------
    # Candidate filtering
    # -----------------------------
    def __get_valid_candidate_indices(evaluation_sample: dict) -> list[int]:
        ignore = config.get("ignore_score", None)
        if ignore is None:
            raise ValueError("config['ignore_score'] is required.")

        eval_field = config["target_fields"]["evaluation"]
        if eval_field not in evaluation_sample:
            raise ValueError(f"evaluation sample does not have field '{eval_field}'.")

        k = len(evaluation_sample[eval_field])
        if k <= 0:
            raise ValueError("Each sample must have at least 1 candidate.")

        score_fields = _score_names()

        valid: list[int] = []
        for i in range(k):
            bad = False
            for score_name in score_fields:
                if score_name not in evaluation_sample:
                    raise ValueError(f"evaluation sample missing score field '{score_name}'.")
                v = evaluation_sample[score_name][i]
                if v == ignore:
                    bad = True
                    break
            if not bad:
                valid.append(i)

        if len(valid) < 2:
            raise ValueError(
                f"Not enough valid candidates after ignoring ignore_score. "
                f"valid={len(valid)} (<2)."
            )
        return valid

    # -----------------------------
    # Preprocessing (standardize)
    # -----------------------------
    def __standardize(vals: list[float]) -> list[float]:
        m = sum(vals) / len(vals)
        s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
        denom = (s or 1.0)
        return [(v - m) / denom for v in vals]

    _PREPROCS = {
        "standardize": __standardize,
    }

    def __preprocess_scores_on_valid(evaluation_sample: dict, valid: list[int]) -> dict:
        """
        Apply preprocessing for each score_name to values ONLY on valid candidates.
        Store the processed score array back in original length k,
        filling invalid positions with None.
        """
        k = len(evaluation_sample[config["target_fields"]["evaluation"]])

        for score_name in _score_names():
            preproc = _score_preproc(score_name)
            if not preproc:
                # no preprocessing requested: still keep original values but None-out invalid to avoid accidental use
                full = [None] * k
                for i in valid:
                    full[i] = evaluation_sample[score_name][i]
                evaluation_sample[score_name] = full
                continue

            if preproc not in _PREPROCS:
                raise ValueError(f"Unknown preprocessing '{preproc}' for score '{score_name}'.")

            proc_fn = _PREPROCS[preproc]
            valid_vals = [float(evaluation_sample[score_name][i]) for i in valid]
            valid_vals_proc = proc_fn(valid_vals)

            full = [None] * k
            for i, vproc in zip(valid, valid_vals_proc):
                full[i] = vproc
            evaluation_sample[score_name] = full

        return evaluation_sample

    # -----------------------------
    # Weighted scores + pairing
    # -----------------------------
    def __calc_weighted_scores_full(evaluation_sample: dict, valid: list[int]) -> list[float | None]:
        """
        Return weighted_scores in original length k.
        invalid positions are None.
        """
        k = len(evaluation_sample[config["target_fields"]["evaluation"]])
        weighted: list[float | None] = [None] * k

        for i in valid:
            s = 0.0
            for score_name in _score_names():
                w = _score_weight(score_name)
                v = evaluation_sample[score_name][i]
                if v is None:
                    # should never happen for valid i
                    raise RuntimeError(f"Score '{score_name}' at valid index {i} is None.")
                s += w * float(v)
            weighted[i] = s

        return weighted

    def __make_pair_from_full(weighted_scores_full: list[float | None], valid: list[int]) -> dict[str, int]:
        if config.get("pair_strategy") == "best_and_worst":
            chosen_idx = max(valid, key=lambda i: weighted_scores_full[i])   # type: ignore[arg-type]
            rejected_idx = min(valid, key=lambda i: weighted_scores_full[i]) # type: ignore[arg-type]
            return {"chosen": chosen_idx, "rejected": rejected_idx}
        else:
            raise ValueError(f"Invalid pair strategy: {config.get('pair_strategy')}")

    def __add_pair_indexes(evaluation_sample: dict) -> dict:
        valid = __get_valid_candidate_indices(evaluation_sample)
        evaluation_sample = __preprocess_scores_on_valid(evaluation_sample, valid)

        evaluation_sample["weighted_scores"] = __calc_weighted_scores_full(evaluation_sample, valid)
        evaluation_sample["pair_indexes"] = __make_pair_from_full(evaluation_sample["weighted_scores"], valid)

        # optional for debugging / analysis
        evaluation_sample["valid_candidate_indices"] = valid
        return evaluation_sample

    # -----------------------------
    # Generation side: add pair columns
    # -----------------------------
    def __add_pair_columns(
        generation_sample: dict,
        sample_idx: int,
        weighted_scores_list: list[list[float | None]],
        pair_indexes_list: list[dict[str, int]],
    ) -> dict:
        chosen_idx = pair_indexes_list[sample_idx]["chosen"]
        rejected_idx = pair_indexes_list[sample_idx]["rejected"]

        gen_field = config["target_fields"]["generation"]
        gen_candidates = generation_sample[gen_field]

        generation_sample["chosen"] = gen_candidates[chosen_idx]
        generation_sample["rejected"] = gen_candidates[rejected_idx]

        # lengths
        if "prompt" in generation_sample:
            generation_sample["prompt_length"] = len(word_tokenize(generation_sample["prompt"]))
        generation_sample["chosen_length"] = len(word_tokenize(generation_sample["chosen"]))
        generation_sample["rejected_length"] = len(word_tokenize(generation_sample["rejected"]))

        # scores
        ws = weighted_scores_list[sample_idx]
        generation_sample["chosen_score"] = ws[chosen_idx]
        generation_sample["rejected_score"] = ws[rejected_idx]

        # indexes
        generation_sample["pair_indexes"] = pair_indexes_list[sample_idx]

        return generation_sample

    # -----------------------------
    # Run
    # -----------------------------
    processed_evaluation_dataset = evaluation_dataset.map(__add_pair_indexes)

    processed_generation_dataset = generation_dataset.map(
        __add_pair_columns,
        with_indices=True,
        fn_kwargs={
            "pair_indexes_list": processed_evaluation_dataset["pair_indexes"],
            "weighted_scores_list": processed_evaluation_dataset["weighted_scores"],
        },
    )

    return processed_generation_dataset, processed_evaluation_dataset

In [ ]:
proc_gen_ds, proc_evl_ds = make_pairs(gen_ds, evl_ds, dataset_config)

In [ ]:
from nltk.tokenize import word_tokenize

def convert_to_dpo_format(sample: dict) -> dict:
    def __convert_to_conversation_format(role: str, text: str) -> dict[str, str]:
        return [{"role": role, "content": text}]
    sample["prompt"] = __convert_to_conversation_format("user", sample["filled_prompt"])
    sample["chosen"] = __convert_to_conversation_format("assistant", sample["chosen"])
    sample["rejected"] = __convert_to_conversation_format("assistant", sample["rejected"])
    return sample

proc_gen_ds = proc_gen_ds.map(convert_to_dpo_format)
proc_gen_ds[0]

In [ ]:
import json
dataset_config_dump = json.dumps(dataset_config)
generation_config = gen_ds["config"][0]
evaluation_config = evl_ds["config"][0]

In [ ]:
def set_configs(sample: dict, dataset_config: str, generation_config: str, evaluation_config: str) -> dict:
    sample["dataset_config"] = dataset_config
    sample["generation_config"] = generation_config
    sample["evaluation_config"] = evaluation_config
    return sample

proc_gen_ds = proc_gen_ds.map(set_configs, fn_kwargs={"dataset_config": dataset_config_dump, "generation_config": generation_config, "evaluation_config": evaluation_config})
proc_gen_ds[0]

In [ ]:
dpo_ds = proc_gen_ds.remove_columns(list(set(proc_gen_ds.column_names) - set(dataset_config["use_columns"])))

In [ ]:
train_size, test_size = dataset_config["data_balance"]["for_training"], dataset_config["data_balance"]["for_evaluation"]
assert train_size + test_size == 1

if test_size > 0:
    split_dpo_ds = dpo_ds.train_test_split(train_size=train_size, seed=dataset_config["data_balance"]["seed"])
    print(f"train_size: {len(split_dpo_ds['train']):,}, test_size: {len(split_dpo_ds['test']):,}")
else:
    split_dpo_ds = dpo_ds

In [ ]:
from datetime import datetime

model_name = "qwen3_4b"
dataset_name = f"{dataset_config['dataset_name']}_{model_name}_{datetime.now().strftime('%Y%m%d%H%M')}"

split_dpo_ds.save_to_disk(os.path.join(
    os.environ["ROOT_DIR"], 
    "dpo_data/data/",
    dataset_name
    ))